In [3]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


## Load Data

In [2]:
source_path = '/content/drive/MyDrive/Colab Notebooks/personal_projects/e_commerce_recsys/meta'
target_path = '/content/drive/MyDrive/UTS Courses/NLP-proj/metadata/'

In [3]:
import shutil
import os

# Check if target_path exists and remove it to allow overwriting
if os.path.exists(target_path):
    print(f"Removing existing target directory: {target_path}")
    shutil.rmtree(target_path)

# Copy the directory
shutil.copytree(source_path, target_path)
print(f"Successfully copied '{source_path}' to '{target_path}'")

Removing existing target directory: /content/drive/MyDrive/UTS Courses/NLP-project
Successfully copied '/content/drive/MyDrive/Colab Notebooks/personal_projects/e_commerce_recsys/meta' to '/content/drive/MyDrive/UTS Courses/NLP-project'


## RAG Vector Database Construction

### Filter and Merge Datasets to One .parquet file

In [1]:
import pyarrow as pa
import pyarrow.parquet as pq
import os

# 2. Define the directory path (matches your Colab Google Drive path)
path = '/content/drive/MyDrive/UTS Courses/NLP-proj/metadata/'

# Get the file list in path folder
file_list = os.listdir(path)

# Filter out non-parquet files
file_list = [file for file in file_list if file.endswith('.parquet')]

# Define the schema with the exact columns and types requested
schema = pa.schema([
    ('main_category', pa.string()),
    ('title', pa.string()),
    ('average_rating', pa.float64()),
    ('features', pa.list_(pa.string())),
    ('description', pa.list_(pa.string())),
    ('price', pa.float64()),
    ('store', pa.string()),
    ('categories', pa.list_(pa.string())),
    ('details', pa.string())
])


tables = []

for file_name in file_list:
    file_path = os.path.join(path, file_name)

    try:
        # Read the parquet file
        table = pq.read_table(file_path)

        # Select only the columns defined in the target schema
        selected_columns = [field.name for field in schema]
        table = table.select(selected_columns)

        # Cast table to enforce strict schema types
        table = table.cast(schema)

        tables.append(table)
        print(f"Successfully loaded: {file_name}")
        del table
    except FileNotFoundError:
        print(f"Warning: File not found at {file_path}")
    except Exception as e:
        print(f"Error processing {file_name}: {e}")




Successfully loaded: All_Beauty_meta.parquet
Successfully loaded: Amazon_Fashion_meta.parquet
Successfully loaded: Appliances_meta.parquet
Successfully loaded: Arts_Crafts_and_Sewing_meta.parquet
Successfully loaded: Automotive_meta.parquet
Successfully loaded: Baby_Products_meta.parquet
Successfully loaded: Beauty_and_Personal_Care_meta.parquet
Successfully loaded: Books_meta.parquet
Successfully loaded: CDs_and_Vinyl_meta.parquet
Successfully loaded: Cell_Phones_and_Accessories_meta.parquet
Successfully loaded: Clothing_Shoes_and_Jewelry_meta.parquet
Successfully loaded: Digital_Music_meta.parquet
Successfully loaded: Electronics_meta.parquet
Successfully loaded: Gift_Cards_meta.parquet
Successfully loaded: Grocery_and_Gourmet_Food_meta.parquet
Successfully loaded: Handmade_Products_meta.parquet
Successfully loaded: Health_and_Household_meta.parquet
Successfully loaded: Health_and_Personal_Care_meta.parquet
Successfully loaded: Home_and_Kitchen_meta.parquet
Successfully loaded: Indus

In [3]:
output_path = '/content/drive/MyDrive/UTS Courses/NLP-proj/metadata/merged_data.parquet'

# Check if any tables were loaded
if tables:
    # Concatenate all the loaded tables
    merged_table = pa.concat_tables(tables)

    # Write to the destination file
    pq.write_table(merged_table, output_path)
    print(f"\nMerged {len(tables)} files into: {output_path}")
else:
    print("\nNo tables to merge.")



Merged 34 files into: /content/drive/MyDrive/UTS Courses/NLP-proj/metadata/merged_data.parquet


### Inspect the final dataset

In [4]:
import os
import pyarrow.parquet as pq
import pandas as pd

# Define the file path
file_path = '/content/drive/MyDrive/UTS Courses/NLP-proj/metadata/merged_data.parquet'

# 1. Check File Size
if os.path.exists(file_path):
    file_size_bytes = os.path.getsize(file_path)
    file_size_mb = file_size_bytes / (1024 * 1024)
    print(f"📁 File Size: {file_size_mb:.2f} MB\n")
else:
    print("❌ File not found. Please ensure the merge was successful.\n")

# 2. Check Schema and Number of Rows using PyArrow
parquet_file = pq.ParquetFile(file_path)

print("📋 Schema:")
print(parquet_file.schema_arrow)
print("-" * 50)

print(f"📊 Number of Rows: {parquet_file.metadata.num_rows:,}\n")
print("-" * 50)


📁 File Size: 10677.91 MB

📋 Schema:
main_category: string
title: string
average_rating: double
features: list<element: string>
  child 0, element: string
description: list<element: string>
  child 0, element: string
price: double
store: string
categories: list<element: string>
  child 0, element: string
details: string
--------------------------------------------------
📊 Number of Rows: 12,206,660

--------------------------------------------------


In [3]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# The parquet_file object is already available from previous cells.

# Define how many rows we want to inspect
num_rows_to_inspect = 10
df_merged = pd.DataFrame() # Initialize an empty DataFrame

# Iterate through batches to get the first N rows efficiently
batches_list = []
current_rows = 0

for batch in parquet_file.iter_batches():
    batches_list.append(batch)
    current_rows += batch.num_rows
    if current_rows >= num_rows_to_inspect:
        break

if batches_list:
    # Concatenate the batches into a single PyArrow Table
    combined_table = pa.Table.from_batches(batches_list)
    # Convert to pandas and take the exact number of rows requested
    df_merged = combined_table.to_pandas().head(num_rows_to_inspect)
else:
    print("No data found in the Parquet file.")


# Display the first 10 rows (which are all the rows loaded)
print("First 10 rows of the merged dataset (loaded efficiently):")
display(df_merged)

First 10 rows of the merged dataset (loaded efficiently):


,main_category,title,average_rating,features,description,price,store,categories,details
0,All Beauty,"Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4...",3.1,[],[],-1.0,Cherioll,[],"{""Brand"": ""Cherioll"", ""Item Form"": ""Powder"", ""..."
1,All Beauty,Stain Bonnet For Baby Bonnet Silk Sleep Cap Fo...,4.1,[],[],-1.0,Edoneery,[],"{""Brand"": ""Edoneery"", ""Material"": ""Silk"", ""Num..."
2,All Beauty,50 Pieces False Eyelash Packaging Box Empty Ey...,3.8,[],[],-1.0,Maitys,[],"{""Package Dimensions"": ""14.49 x 11.26 x 2.36 i..."
3,All Beauty,4 Pieces Satin Bonnet Adjustable Sleep Cap Dou...,4.3,[],[],-1.0,Geyoga,[],"{""Package Dimensions"": ""12.49 x 9.97 x 1.46 in..."
4,All Beauty,"Stainless Steel Beard Comb, Portable Fashionab...",4.1,[],[],-1.0,Brrnoo,[],"{""Material"": ""Metal"", ""Brand"": ""Brrnoo"", ""Hair..."
5,All Beauty,Nadula Hair 10A Brazilian Body Wave U Part Hum...,3.6,[],[],-1.0,Nadula,[],"{""Material"": ""Human Hair"", ""Hair Type"": ""Curly..."
6,All Beauty,VIROCHEMISTRY Pheromones For Women (Elixir) - ...,3.7,[🗽 SCIENTIFICALLY PROVEN! Scientifically Formu...,[The Most Amazingly Effective and Fantastic Sm...,29.8,VIROCHEMISTRY,[],"{""Brand"": ""VIROCHEMISTRY"", ""Item Form"": ""Liqui..."
7,All Beauty,10 PCS Professional Hair Cutting Scissors 6.7i...,4.3,[],[],-1.0,Astraet,[],"{""Color"": ""Pink"", ""Material"": ""Rubber"", ""Brand..."
8,All Beauty,BEWAVE Hair Brush Sponge Twist With Comb Hair ...,4.2,[],[],-1.0,BEWAVE,[],"{""Brand"": ""BEWAVE"", ""Shape"": ""Oblong"", ""Unit C..."
9,All Beauty,"Zydeco Chop Chop Cajun Seasoning Base, 8 Ounce...",4.7,"[All Natural blend of Dehydrated Onion, Dehydr...",[Zydeco Chop Chop is a blend of Dehydrated Oni...,-1.0,BORELTH,[],"{""Package Dimensions"": ""9.61 x 7.17 x 3.07 inc..."


In [6]:
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   main_category   10 non-null     object 
 1   title           10 non-null     object 
 2   average_rating  10 non-null     float64
 3   features        10 non-null     object 
 4   description     10 non-null     object 
 5   price           10 non-null     float64
 6   store           10 non-null     object 
 7   categories      10 non-null     object 
 8   details         10 non-null     object 
dtypes: float64(2), object(7)
memory usage: 852.0+ bytes


### Data Formatting

#### Find the Top 100 keys
First, I need to identify the top 100 most frequently occurring keys, which will serve as the keys used in the final text consolidation process.

In [1]:
import pyarrow.parquet as pq
import json
import ast
from collections import Counter

file_path = '/content/drive/MyDrive/UTS Courses/NLP-proj/metadata/merged_data.parquet'

# Load the parquet file metadata and initialize a Counter to store key frequencies
parquet_file = pq.ParquetFile(file_path)
key_counts = Counter()

# Process row groups one by one to keep memory usage low
for i in range(parquet_file.num_row_groups):
    # Read only the 'details' column for the current row group
    table = parquet_file.read_row_group(i, columns=['details'])
    details_column = table['details'].to_pylist()

    for item in details_column:
        if item is not None and isinstance(item, str):
            try:
                # Parse the string into a dictionary
                item_dict = json.loads(item)
            except (ValueError, TypeError):
                # Fallback to literal_eval if single quotes are used instead of double quotes
                try:
                    item_dict = ast.literal_eval(item)
                except Exception:
                    continue

            # Update the counts for all keys present in the instance
            if isinstance(item_dict, dict):
                for key in item_dict.keys():
                    key_counts[key] += 1

# Display the top 100 most frequent keys
print("Top 100 keys and their appearance counts:")
top_100 = key_counts.most_common(100)
for rank, (key, count) in enumerate(top_100, 1):
    print(f"{rank}. {key}: {count}")

Top 100 keys and their appearance counts:
1. Date First Available: 8291307
2. Manufacturer: 7420305
3. Item Weight: 6562242
4. Brand: 6458756
5. Best Sellers Rank: 5175938
6. Product Dimensions: 4515134
7. Item model number: 4459872
8. Package Dimensions: 4246006
9. Color: 4016154
10. Material: 3829111
11. Is Discontinued By Manufacturer: 2572628
12. Department: 2237939
13. Special Feature: 1749042
14. Style: 1746336
15. Size: 1387637
16. Language: 1284816
17. Shape: 1233457
18. Country of Origin: 1213505
19. Number of Items: 1185959
20. Manufacturer Part Number: 1018132
21. Unit Count: 990632
22. Age Range (Description): 987907
23. Publisher: 982996
24. Theme: 948077
25. Included Components: 884925
26. Number of Pieces: 878206
27. Pattern: 826371
28. Part Number: 825974
29. Item Dimensions LxWxH: 820499
30. Batteries Required?: 737655
31. Product Care Instructions: 687376
32. ISBN 13: 682551
33. Mounting Type: 680275
34. Dimensions: 638940
35. Finish Type: 594883
36. ISBN 10: 562625
3

In [4]:
# Save the Top 100 keys to one list with low case
# exclude the existed keys and filter the zero length key
# Convert all the keys to lowcase
exist_keys = list(df_merged.columns)
exist_keys.remove('details')
top_100_keys = [key.lower() for key, _ in top_100 if key.lower() not in exist_keys and len(key)>0]
final_keys = exist_keys + top_100_keys
print(final_keys)
print(f"final keys number: {len(final_keys)}")

['main_category', 'title', 'average_rating', 'features', 'description', 'price', 'store', 'categories', 'date first available', 'manufacturer', 'item weight', 'brand', 'best sellers rank', 'product dimensions', 'item model number', 'package dimensions', 'color', 'material', 'is discontinued by manufacturer', 'department', 'special feature', 'style', 'size', 'language', 'shape', 'country of origin', 'number of items', 'manufacturer part number', 'unit count', 'age range (description)', 'publisher', 'theme', 'included components', 'number of pieces', 'pattern', 'part number', 'item dimensions lxwxh', 'batteries required?', 'product care instructions', 'isbn 13', 'mounting type', 'dimensions', 'finish type', 'isbn 10', 'batteries included?', 'occasion', 'room type', 'item form', 'item package quantity', 'batteries required', 'upc', 'publication date', 'paperback', 'sticky notes', 'file size', 'text to speech', 'enhanced typesetting', 'word wise', 'print length', 'special features', 'x ray

#### Define the final format and create final text dataset
First, I define the final text structure: I specify all keys, with each key and its corresponding content separated by a colon, and each key‑content pair separated by a semicolon.

After finalizing this file structure, I construct the final text dataset. Specifically, I add a new column to the original dataset, which contains plain text generated by merging all columns of each row into a single entry.

We will limite the length of 'features' and 'description':
- Set the leng of content in 'features' and 'description' as 300 tokens max for each and drop the remain string

In [1]:
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
import json
import ast
import gc

# ── Configuration ──────────────────────────────────────────────────────────────
KEY_LIST = ['date first available', 'manufacturer', 'item weight', 'brand', 'best sellers rank', 'product dimensions', 'item model number', 'package dimensions', 'color', 'material', 'is discontinued by manufacturer', 'department', 'special feature', 'style', 'size', 'language', 'shape', 'country of origin', 'number of items', 'manufacturer part number', 'unit count', 'age range (description)', 'publisher', 'theme', 'included components', 'number of pieces', 'pattern', 'part number', 'item dimensions lxwxh', 'batteries required?', 'product care instructions', 'isbn 13', 'mounting type', 'dimensions', 'finish type', 'isbn 10', 'batteries included?', 'occasion', 'room type', 'item form', 'item package quantity', 'batteries required', 'upc', 'publication date', 'paperback', 'sticky notes', 'file size', 'text to speech', 'enhanced typesetting', 'word wise', 'print length', 'special features', 'x ray', 'brand name', 'recommended uses for product', 'package weight', 'screen reader', 'compatible devices', 'power source', 'form factor', 'other display features', 'item package dimensions l x w x h', 'manufacturer recommended age', 'special features', 'fabric type', 'model name', 'closure type', 'compatible phone models', 'voltage', 'specific uses for product', 'vehicle service type', 'material type', 'batteries', 'model', 'capacity', 'connectivity technology', 'oem part number', 'wattage', 'simultaneous device usage', 'assembly required', 'item dimensions  lxwxh', 'installation type', 'frame material', 'seasons', 'number of discs', 'finish', 'indoor/outdoor usage', 'release date', 'reusability', 'scent', 'exterior', 'whats in the box', 'warranty description', 'cartoon character', 'light source type', 'suggested users', 'handle material', 'is dishwasher safe', 'studio']

INPUT_FILE  = "/content/drive/MyDrive/UTS Courses/NLP-proj/metadata/All_Beauty_meta.parquet"   # ← change to your actual file path
OUTPUT_FILE = "/content/drive/MyDrive/UTS Courses/NLP-proj/All_Beauty_meta.parquet_prep"
CHUNK_SIZE  = 100_000           # rows per chunk; tune to your available RAM


# ── Per-row processing ─────────────────────────────────────────────────────────
def parse_list_field(value):
    """Safely parse a field that should be a list of strings."""
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        try:
            result = ast.literal_eval(value)
            if isinstance(result, list):
                return result
        except Exception:
            pass
        return [value]
    return []


def parse_details(value):
    """Safely parse the details JSON/dict field."""
    if isinstance(value, dict):
        return value
    if isinstance(value, str):
        # Try JSON first, then ast.literal_eval
        for parser in (json.loads, ast.literal_eval):
            try:
                result = parser(value)
                if isinstance(result, dict):
                    return result
            except Exception:
                continue
    return {}


def build_text(row):
    parts = []

    # 1. main_category
    parts.append(f"main_category:{row.get('main_category', 'N/A')}")

    # 2. title
    parts.append(f"title:{row.get('title', 'N/A')}")

    # 3. average_rating
    parts.append(f"average_rating:{row.get('average_rating', 'N/A')}")

    # 4. features — list → concat → first 1200 chars
    features = parse_list_field(row.get('features', []))
    if features:
        parts.append(f"features: {''.join(str(s) for s in features)[:1200]}")
    else:
        parts.append("features:N/A")

    # 5. description — list → concat → first 1200 chars
    description = parse_list_field(row.get('description', []))
    if not description:
        parts.append(f"description:{''.join(str(s) for s in description)[:1200]}")
    else:
        parts.append("description:N/A")

    # 6. price — replace -1 with N/A
    price = row.get('price', 'N/A')
    try:
        price = 'N/A' if float(price) == -1 else price
    except (TypeError, ValueError):
        pass
    parts.append(f"price:{price}")

    # 7. store
    parts.append(f"store:{row.get('store', 'N/A')}")

    # 8. categories — list → concat → first 200 chars
    categories = parse_list_field(row.get('categories', 'N/A'))
    if not categories:
        parts.append("categories:N/A")
    else:
        parts.append(f"categories: {''.join(str(s) for s in categories)[:200]}")

    # 9. details — JSON string → dict → key_list lookup
    details = parse_details(row.get('details', {}))
    detail_parts = []
    for k, v in details.items():
      k_lower = k.lower()
      if k_lower in KEY_LIST:
          detail_parts.append(f"{k}:{v}")
      else:
          detail_parts.append(f"{k}:N/A")
      parts.append("|".join(detail_parts))

    return "|".join(parts)

# Chunk preprocess
def process_chunk(df: pd.DataFrame) -> pd.DataFrame:
    df["text"] = df.apply(build_text, axis=1)
    return df[["text"]]

# ── Chunked read → process → write
def process(input_path: str, output_path: str, chunk_size: int = CHUNK_SIZE):
    parquet_file = pq.ParquetFile(input_path)
    total_rows   = parquet_file.metadata.num_rows
    print(f"Total rows: {total_rows:,}  |  Chunk size: {chunk_size:,}")

    writer    = None
    rows_done = 0

    try:
        for batch in parquet_file.iter_batches(batch_size=chunk_size):
            # 1. Arrow batch → pandas
            df = batch.to_pandas()
            del batch          # free the Arrow batch immediately
            gc.collect()

            # 2. Process
            result = process_chunk(df)
            del df             # free the full-width chunk; only 'text' survives
            gc.collect()

            # 3. pandas → Arrow table → write
            table = pa.Table.from_pandas(result, preserve_index=False)
            del result         # free pandas frame before writing
            gc.collect()

            if writer is None:
                writer = pq.ParquetWriter(output_path, table.schema)
            writer.write_table(table)

            rows_done += table.num_rows
            del table
            gc.collect()

            print(f"  {rows_done:>12,} / {total_rows:,} rows processed "
                  f"({100 * rows_done / total_rows:.1f}%)")

    finally:
        if writer:
            writer.close()

    print(f"\nDone. Output saved to: {output_path}")



# process(INPUT_FILE, OUTPUT_FILE)

In [2]:
process(INPUT_FILE, OUTPUT_FILE)

Total rows: 44,634  |  Chunk size: 100,000
        44,634 / 44,634 rows processed (100.0%)

Done. Output saved to: /content/drive/MyDrive/UTS Courses/NLP-proj/All_Beauty_meta.parquet_prep


In [3]:
# TEST
df_text = pd.read_parquet(OUTPUT_FILE)
# print the first row
print(df_text['text'][0])

main_category:All Beauty|title:Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4D Imitation Eyebrow Tattoos, 4D Hair-like Authentic Eyebrows Waterproof Long Lasting for Woman & Man Makeup Tool|average_rating:3.1|features:N/A|description:|price:N/A|store:Cherioll|categories:N/A|Brand:Cherioll|Brand:Cherioll|Item Form:Powder|Brand:Cherioll|Item Form:Powder|Finish Type:Natural|Brand:Cherioll|Item Form:Powder|Finish Type:Natural|Product Benefits:N/A|Brand:Cherioll|Item Form:Powder|Finish Type:Natural|Product Benefits:N/A|Skin Type:N/A|Brand:Cherioll|Item Form:Powder|Finish Type:Natural|Product Benefits:N/A|Skin Type:N/A|Package Dimensions:8.43 x 5.91 x 0.87 inches; 8.78 Ounces|Brand:Cherioll|Item Form:Powder|Finish Type:Natural|Product Benefits:N/A|Skin Type:N/A|Package Dimensions:8.43 x 5.91 x 0.87 inches; 8.78 Ounces|Item model number:eyebrow sticker001


In [4]:
# ------- Final RUN ---------
KEY_LIST = ['date first available', 'manufacturer', 'item weight', 'brand', 'best sellers rank', 'product dimensions', 'item model number', 'package dimensions', 'color', 'material', 'is discontinued by manufacturer', 'department', 'special feature', 'style', 'size', 'language', 'shape', 'country of origin', 'number of items', 'manufacturer part number', 'unit count', 'age range (description)', 'publisher', 'theme', 'included components', 'number of pieces', 'pattern', 'part number', 'item dimensions lxwxh', 'batteries required?', 'product care instructions', 'isbn 13', 'mounting type', 'dimensions', 'finish type', 'isbn 10', 'batteries included?', 'occasion', 'room type', 'item form', 'item package quantity', 'batteries required', 'upc', 'publication date', 'paperback', 'sticky notes', 'file size', 'text to speech', 'enhanced typesetting', 'word wise', 'print length', 'special features', 'x ray', 'brand name', 'recommended uses for product', 'package weight', 'screen reader', 'compatible devices', 'power source', 'form factor', 'other display features', 'item package dimensions l x w x h', 'manufacturer recommended age', 'special features', 'fabric type', 'model name', 'closure type', 'compatible phone models', 'voltage', 'specific uses for product', 'vehicle service type', 'material type', 'batteries', 'model', 'capacity', 'connectivity technology', 'oem part number', 'wattage', 'simultaneous device usage', 'assembly required', 'item dimensions  lxwxh', 'installation type', 'frame material', 'seasons', 'number of discs', 'finish', 'indoor/outdoor usage', 'release date', 'reusability', 'scent', 'exterior', 'whats in the box', 'warranty description', 'cartoon character', 'light source type', 'suggested users', 'handle material', 'is dishwasher safe', 'studio']

INPUT_FILE  = "/content/drive/MyDrive/UTS Courses/NLP-proj/metadata/merged_data.parquet"   # ← change to your actual file path
OUTPUT_FILE = "/content/drive/MyDrive/UTS Courses/NLP-proj/final_text_data.parquet"
CHUNK_SIZE  = 500000           # rows per chunk; tune to your available RAM

process(INPUT_FILE, OUTPUT_FILE)

Total rows: 12,206,660  |  Chunk size: 100,000
       100,000 / 12,206,660 rows processed (0.8%)
       200,000 / 12,206,660 rows processed (1.6%)
       300,000 / 12,206,660 rows processed (2.5%)
       400,000 / 12,206,660 rows processed (3.3%)
       500,000 / 12,206,660 rows processed (4.1%)
       600,000 / 12,206,660 rows processed (4.9%)
       700,000 / 12,206,660 rows processed (5.7%)
       800,000 / 12,206,660 rows processed (6.6%)
       900,000 / 12,206,660 rows processed (7.4%)
     1,000,000 / 12,206,660 rows processed (8.2%)
     1,100,000 / 12,206,660 rows processed (9.0%)
     1,200,000 / 12,206,660 rows processed (9.8%)
     1,300,000 / 12,206,660 rows processed (10.6%)
     1,400,000 / 12,206,660 rows processed (11.5%)
     1,500,000 / 12,206,660 rows processed (12.3%)
     1,600,000 / 12,206,660 rows processed (13.1%)
     1,700,000 / 12,206,660 rows processed (13.9%)
     1,800,000 / 12,206,660 rows processed (14.7%)
     1,900,000 / 12,206,660 rows processed (15.

In [5]:
import os
import pyarrow.parquet as pq
import pandas as pd

# Define the file path
file_path = '/content/drive/MyDrive/UTS Courses/NLP-proj/final_text_data.parquet'

# 1. Check File Size
if os.path.exists(file_path):
    file_size_bytes = os.path.getsize(file_path)
    file_size_mb = file_size_bytes / (1024 * 1024)
    print(f"📁 File Size: {file_size_mb:.2f} MB\n")
else:
    print("❌ File not found. Please ensure the merge was successful.\n")

# 2. Check Schema and Number of Rows using PyArrow
parquet_file = pq.ParquetFile(file_path)

print("📋 Schema:")
print(parquet_file.schema_arrow)
print("-" * 50)

print(f"📊 Number of Rows: {parquet_file.metadata.num_rows:,}\n")
print("-" * 50)
# Load the 10 rows to inspect

# Define how many rows we want to inspect
num_rows_to_inspect = 10
df = pd.DataFrame() # Initialize an empty DataFrame

# Iterate through batches to get the first N rows efficiently
batches_list = []
current_rows = 0

for batch in parquet_file.iter_batches():
    batches_list.append(batch)
    current_rows += batch.num_rows
    if current_rows >= num_rows_to_inspect:
        break

if batches_list:
    # Concatenate the batches into a single PyArrow Table
    combined_table = pa.Table.from_batches(batches_list)
    # Convert to pandas and take the exact number of rows requested
    df = combined_table.to_pandas().head(num_rows_to_inspect)
else:
    print("No data found in the Parquet file.")


# Display the first 10 rows (which are all the rows loaded)
print("First 10 rows of the merged dataset (loaded efficiently):")
display(df)

📁 File Size: 4193.99 MB

📋 Schema:
text: string
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 183
--------------------------------------------------
📊 Number of Rows: 12,206,660

--------------------------------------------------
First 10 rows of the merged dataset (loaded efficiently):


,text
0,main_category:All Beauty|title:Tattoo Eyebrow ...
1,main_category:All Beauty|title:Stain Bonnet Fo...
2,main_category:All Beauty|title:50 Pieces False...
3,main_category:All Beauty|title:4 Pieces Satin ...
4,main_category:All Beauty|title:Stainless Steel...
5,main_category:All Beauty|title:Nadula Hair 10A...
6,main_category:All Beauty|title:VIROCHEMISTRY P...
7,main_category:All Beauty|title:10 PCS Professi...
8,main_category:All Beauty|title:BEWAVE Hair Bru...
9,main_category:All Beauty|title:Zydeco Chop Cho...
